In [15]:
import pandas as pd
import json
import os
import sys
from dotenv import load_dotenv



In [16]:
# 1. Load your custom cleaning pipeline
sys.path.append(os.path.abspath('..'))
from src.features import process_raw_to_clean

# Load the .env file so Hugging Face can find the HF_TOKEN
from pathlib import Path
env_path = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=env_path)

# NOW we can import the ML libraries safely
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

In [17]:
# 2. Load and clean the data
file_path = '../data/raw/20260213_technology_news.json'
with open(file_path, 'r') as f:
    raw_data = json.load(f)

df_raw = pd.DataFrame(raw_data['articles'])
df_clean = process_raw_to_clean(df_raw)

# BERTopic requires a simple Python list of strings
docs = df_clean['text_cleaned'].tolist()

print(f"Ready to cluster {len(docs)} articles...\n")



Ready to cluster 43 articles...



In [18]:
# 3. Load the Embedding Model
# "all-MiniLM-L6-v2" is highly optimised, fast, and won't crash my Mac's RAM
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1419.63it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
# 4.
from sklearn.feature_extraction.text import CountVectorizer

# 1. Define the Vectorizer to remove english stopwords and look for 1 or 2-word phrases
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2))

# 2. Re-initialize BERTopic with the new vectorizer
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model, # <-- ADDED THIS
    min_topic_size=5
)

# 3. Re-train the model (this will be fast since embeddings are cached)
topics, probs = topic_model.fit_transform(docs)



In [21]:
# 4. View the topics
topic_info = topic_model.get_topic_info()
display(topic_info.head(10))

,Topic,Count,Name,Representation,Representative_Docs
0,-1,16,-1_new_says_official_windows,"[new, says, official, windows, google, adds, r...",[sony wfxm new wireless earbuds star in compar...
1,0,16,0_apple_update_iphone_confirms,"[apple, update, iphone, confirms, stock, galax...",[diablo gets first major update in years wit...
2,1,11,1_game_arc_play reanimal_play,"[game, arc, play reanimal, play, new riot, pla...",[the switch s gameshare multiplayer turns this...
